In [21]:
import os
import re
from pathlib import Path

import pandas as pd

from semevalpolar.finetuning.instruct.predict import generate_predictions_jsonl
from typing import List

from semevalpolar.utils import get_project_root

# Using dev phase data

In [22]:
data_path = Path(os.path.join(get_project_root(), "data-public", "test", "eng.csv"))
dev_df = pd.read_csv(data_path)

In [24]:
records = dev_df["text"]
gold = dev_df["polarization"]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [25]:
import json

labels = []
text = []
with open(f'{get_project_root()}/predictions/final_eval/sft/predictions-test.jsonl', 'r') as f:
	for line in f:
		try:
			json_obj = json.loads(line)
			labels.append(json_obj['extracted_label'])
			text.append(json_obj['prediction'])
		except json.JSONDecodeError:
			pass

In [14]:
text[0]

'Input:\nHow Republicans flipped Americas state supreme\n\nReasoning:\nTarget referenced: Republicans\nClaim type: factual description\nManifestations present:\n- Stereotype: absent\n- Vilification: absent\n- Dehumanization: absent\n- Extreme Language and Absolutism: absent\n- Lack of Empathy or Understanding: absent\n- Invalidation: absent\nDecision basis:\nThe text presents a neutral statement of fact about political events without employing any polarizing language or framing.\n\nFinal Answer:\n0'

In [26]:
comparison = pd.DataFrame({
	"Predictions": labels,
	"Ground Truth": gold,
    "Text": text
})

wrong = comparison[comparison["Predictions"] != comparison["Ground Truth"]]

In [27]:
from templates import evaluate_metrics

evaluate_metrics(comparison, "Ground Truth", "Predictions")

{'accuracy': np.float64(0.7926997245179064),
 'micro_precision': np.float64(0.7447257383966245),
 'micro_recall': np.float64(0.6622889305816135),
 'micro_f1': np.float64(0.7010923535253227),
 'macro_f1': np.float64(0.7712103834047278)}

# Using training dataset

In [ ]:
data_path = Path("data/archive/splits/val.jsonl")
with data_path.open("r", encoding="utf-8") as f:
    records = f.readlines()

In [ ]:
type(records)

In [ ]:
import re
import codecs
from typing import List

INPUT_RE = re.compile(r"Input:\s*(.*?)\s*Reasoning:", re.DOTALL)


def get_all_inputs(records: List[str]) -> List[str]:
    cleaned: List[str] = []

    for text in records:
        m = INPUT_RE.search(text)
        if not m:
            cleaned.append("")
            continue

        # decode escaped sequences like \\n, \\t
        decoded = codecs.decode(m.group(1), "unicode_escape")

        # normalize all whitespace
        normalized = " ".join(decoded.split())

        cleaned.append(normalized)

    return cleaned

In [ ]:
dataset = get_all_inputs(records)

In [ ]:
predictions = generate_predictions_jsonl(dataset)

In [ ]:
import torch

In [ ]:
print(torch.cuda.is_available())

In [ ]:
import json

labels = []
text = []
with open("predictions/predictions.jsonl", "r") as f:
    for line in f:
        try:
            json_obj = json.loads(line)
            labels.append(json_obj["extracted_label"])
            text.append(json_obj["prediction"])
        except json.JSONDecodeError:
            pass

In [ ]:
import json
import re


def extract_final_answers(file_path):
    final_answers = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                json_obj = json.loads(line)
                match = re.search(r"Final Answer:\n(\d+)", json_obj["text"])
                if match:
                    final_answers.append(int(match.group(1)))
            except json.JSONDecodeError:
                pass
    return final_answers


final_answers = extract_final_answers("data/archive/splits/val.jsonl")

In [ ]:
comparison = pd.DataFrame(
    {"Text": text, "Predictions": labels, "Ground Truth": final_answers}
)

In [ ]:
len(comparison[comparison["Predictions"] != comparison["Ground Truth"]])

In [21]:
wrong = comparison[comparison["Predictions"] != comparison["Ground Truth"]]
wrong

,Text,Predictions,Ground Truth
0,Input:\nThe only sober person around this deba...,0,1
1,Input:\nSo The Hill is repeating Russian propa...,1,0
6,Input:\nAn election fraud denier beats an elec...,0,1
12,"Input:\nYes, instead of changing course and em...",0,1
25,Input:\nSo Trump wants a Socialist country wit...,0,1
...,...,...,...
285,Input:\nIm having a hard time deciphering betw...,0,1
292,Input:\nThe streets are flooded with refugees\...,0,1
293,Input:\nWhy Trump Nation voters ignored the li...,0,1
296,Input:\nWith one thing I would disagree corrup...,0,1


# 1 Example

In [ ]:
import re
import json


def extract_labels_jsonl(input_path, output_path):
    pattern = re.compile(r"Final Answer \(output ONLY 0 or 1\):\s*(\d+)")

    with (
        open(input_path, "r", encoding="utf-8") as fin,
        open(output_path, "w", encoding="utf-8") as fout,
    ):
        for line in fin:
            obj = json.loads(line)

            # Prefer extracted_label if it already exists
            if "extracted_label" in obj:
                label = int(obj["extracted_label"])
            else:
                # Fallback: extract from prediction text
                text = obj.get("prediction", "")
                match = pattern.search(text)
                if match is None:
                    continue  # or raise an error if required
                label = int(match.group(1))

            fout.write(json.dumps({"label": label}) + "\n")

In [ ]:
extract_labels_jsonl(
    os.path.join(
        get_project_root(),
        "src",
        "semevalpolar",
        "finetuning",
        "instruct",
        "predictions",
        "predictions.jsonl",
    ),
    os.path.join(
        get_project_root(),
        "src",
        "semevalpolar",
        "finetuning",
        "instruct",
        "predictions",
        "gold.jsonl",
    ),
)

In [ ]:
def predict(text):
    prompt = f"""Input:
            {text}

            Reasoning:

            """